# RSS Feed creation

## Connect to a Velocity instance
For this example, we will connect to `a4iot-dev` 

In [ ]:
from arcgis import GIS
from arcgis.realtime.velocity.feeds_manager import Feed

gis = GIS(
    url="https://devext.arcgis.com",
    username="pythontest_a4iot",
    password="v3locity.pyth0n",
)

velocity = gis.velocity

feeds = gis.velocity.feeds
feeds

## Configure the RSS Feed
Configuring a non-GeoRSS feed that contains location and info about recent earthquakes.

Feed Location: https://web.a4iot.com/RSS/usgs_non_georss_float.xml

In [ ]:
from arcgis.realtime.velocity.feeds import RSS
from arcgis.realtime.velocity.http_authentication_type import (
    NoAuth,
    BasicAuth,
    CertificateAuth,
)
from arcgis.realtime.velocity.input.format import GeoRssFormat
from arcgis.realtime.velocity.feeds.geometry import XYZGeometry, SingleFieldGeometry
from arcgis.realtime.velocity.feeds.time import TimeInterval, TimeInstant
from arcgis.realtime.velocity.feeds.run_interval import RunInterval


# RSS Properties
name = "rss_feed_1"
description = "some description about the rss feed"
url = "https://web.a4iot.com/RSS/usgs_non_georss_float.xml"
http_auth = NoAuth()
# http_auth = BasicAuth(username="user1", password="123")
# http_auth = CertificateAuth(pfx_file_http_location="https://some.where", password="123")

http_headers = {}
# http_headers = {
#     "Content-Type": "application/json"
# }

# data_format = GeoRssFormat()

rss = RSS(
    label=name,
    description=description,
    rss_url=url,
    http_auth_type=http_auth,
    http_headers=http_headers,
    data_format=None,
)



# # all properties can also be defined right away in the constructor as follows
# data_format = GeoRssFormat()
# geometry = XYZGeometry(
#     x_field="category_longitude",
#     y_field="category_latitude",
#     wkid=4326,
#     z_field="category_altitude",
#     z_unit="Meters"
# )
#
# time = TimeInterval(
#     interval_start_field="pubDate",
#     interval_end_field="updated"
# )
#
# run_interval = RunInterval(
#     cron_expression="0 * * ? * * *",
#     timezone="America/Los_Angeles"
# )
#
# rss1 = RSS(
#     rss_url=url,
#     http_auth_type=http_auth,
#     http_headers=http_headers,
#     label=name,
#     description=description,
#     track_id_field="link",
#     data_format=data_format,
#     geometry=geometry,
#     time=time,
#     run_interval=run_interval
# )

### Manipulate the schema - rename or remove fields, change field data-type
1. Renaming `title` to `updated_field`
2. Dropping `description`

In [ ]:
rss.rename_field("title", "updated_field")
rss.remove_field("description")

### Set track id field
Set `link` as the track-id (Sorry, this feed doesn't have a good candidate for track-id!)

In [ ]:
rss.set_track_id("link")

### Set time field
Time interval
1. Start time - `pubDate`
2. End time - `updated`

In [ ]:
# time interval
time = TimeInterval(interval_start_field="pubDate", interval_end_field="updated")

# time instant
# time = TimeInstant(time_field="pubDate")


rss.set_time_config(time=time)

### Set geometry field
Configuring X,Y and Z fields

In [ ]:
geometry = XYZGeometry(
    x_field="category_longitude",
    y_field="category_latitude",
    wkid=4326,
    z_field="category_altitude",
    z_unit="Meters",
)

# a single field geometry could also be configured
# geometry = SingleFieldGeometry(
#     geometry_field="",
#     geometry_type="esriGeometryPoint",
#     geometry_format="esrijson",
#     wkid=4326
# )

rss.set_geometry_config(geometry=geometry)

### Set recurrence

In [ ]:
rss.run_interval = RunInterval(
    cron_expression="0 * * ? * * *", timezone="America/Los_Angeles"
)

### Create the Feed!

In [ ]:
feeds.create(rss)

feeds.items